# Assignment 5 — Can We Run This On Our Own Hardware?

**ISM 6564 — Text Analytics · Fall 2026 · Week 5: Transformers & Language Models**

Northwind Outfitters' support desk takes about 2,400 tickets a week and a human reads
every one before it lands in a queue. The operations director wants the first pass
automated and has two proposals: call a frontier model's API for every ticket, or run a
small open-weights model on the Mac Studio already in the ops room, so customer text never
leaves the building.

Nobody in the room knows what a half-billion-number model can actually do. You are going
to find out, in numbers, on the desk's own tickets — the two bills it costs to run, what it
does with a message, how often it routes correctly, and how that compares with the
frontier option. Then you write the recommendation.

Read [`README.md`](README.md) for the task descriptions and the rubric, and
[`../data/README.md`](../data/README.md) for how the ticket set was labelled and what the
frontier fixture is and is not.

---

## How to work through this notebook

- Every place you have to write something is marked `# TODO`. Everything else — imports,
  loading, the chat-prompt helper, the metric tables, the cost constants — is written for
  you.
- **The notebook runs top to bottom right now, with nothing filled in.** Cells that depend
  on an unfinished `# TODO` print `SKIPPED` and move on. As you fill them in, more of the
  notebook comes alive. Re-run from the top when you finish a task.
- Markdown cells headed **Your answer** are graded. Replace the placeholder text. Tasks 3
  and 4 put more marks on the prose than on the code.
- Do not change `SEED`, and do not change `GREEDY`.

Every task below names the lecture slide it comes from. If a step feels unfamiliar, open the
companion notes for that slide and the lecture notebook section beside it before you ask an
AI assistant.

### Before you start

The first run downloads about **6 GB** of model weights into `~/.cache/huggingface/hub`
(four checkpoints). Do that on a connection you trust and not at 11 PM on the 29th. After
the first run everything is read from disk. **Google Colab** works too: pick a T4 runtime;
the downloads take about a minute per session.


## Setup

Seeds are set here for `random`, `numpy` and `torch`. Greedy decoding is deterministic
(lecture: *Choosing the next token*), so your **accuracy** numbers should reproduce exactly
on the same machine. Your **timings** will not reproduce, on your machine or anyone else's —
that is expected, and it is why the write-ups ask you to explain directions and ratios
rather than defend three decimal places.


In [1]:
# --- imports, seeds, device --------------------------------------------------
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

SEED = 6564
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"torch {torch.__version__}")
print(f"device: {device}")


torch 2.11.0+cpu
device: cpu


### The models, and the decoding settings

Three instruction-tuned decoders of three sizes — the lecture's size ladder, one rung
smaller at the bottom — plus the **base** version of the middle one, so Task 2 can show what
instruction tuning changed.

`GREEDY` pins the decoding settings. A checkpoint ships its own defaults (Qwen's turn
sampling on, with a temperature of 0.7), and `generate()` applies them unless you say
otherwise. Greedy means: always the most likely token, so the same prompt gives the same
text every time. Use `GREEDY` everywhere except where Task 2 tells you to sample.


In [2]:
# --- models and pinned decoding --------------------------------------------------
MODELS = [
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
]
BASE = "Qwen/Qwen2.5-0.5B"          # the same weights as MODELS[1], before instruction tuning
DTYPE = torch.float32               # full precision: 4 bytes per learned number, as in the lecture

GREEDY = dict(do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0)


def SAMPLE(temperature: float, top_p: float = 1.0) -> dict:
    """Sampling settings: temperature, optional top-p, and nothing else (top-k off)."""
    return dict(do_sample=True, temperature=temperature, top_p=top_p, top_k=0, repetition_penalty=1.0)


# What each checkpoint would do if you did NOT override it:
for _m in MODELS:
    _gc = GenerationConfig.from_pretrained(_m)
    print(f"{_m:42s} do_sample={_gc.do_sample}  temp={_gc.temperature}  "
          f"top_p={_gc.top_p}  top_k={_gc.top_k}  rep_pen={_gc.repetition_penalty}")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

HuggingFaceTB/SmolLM2-360M-Instruct        do_sample=None  temp=None  top_p=None  top_k=None  rep_pen=None


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-0.5B-Instruct                 do_sample=True  temp=0.7  top_p=0.8  top_k=20  rep_pen=1.1


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-1.5B-Instruct                 do_sample=True  temp=0.7  top_p=0.8  top_k=20  rep_pen=1.1


In [3]:
# --- scaffolding so this notebook runs with TODOs unfilled -------------------
# You do not need to modify anything in this cell.

class Unfinished(Exception):
    """Raised by scaffolding you have not filled in yet."""


def todo(what: str):
    """Call from inside an unfinished TODO so the notebook still runs."""
    raise Unfinished(f"{what} is not implemented yet.")


def attempt(label, fn, *args, **kwargs):
    """Run `fn`; return None (with a message) if it hits an unfinished TODO."""
    try:
        return fn(*args, **kwargs)
    except Unfinished as exc:
        print(f"SKIPPED [{label}]  {exc}  Fill in the TODO above and re-run.")
        return None


def need(*values) -> bool:
    """True (with a message) if any earlier TODO is still unfilled."""
    if any(v is None for v in values):
        print("SKIPPED  an earlier `# TODO` is still unfilled. Finish it, then re-run this cell.")
        return True
    return False


def sync():
    """Block until the accelerator has actually finished. Timing is a lie without it."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def free_model(model=None):
    """Drop a model and hand the memory back, so four checkpoints fit in one run."""
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()


def load_model(name: str, dtype=DTYPE):
    """Provided. Returns (tokenizer, model) on `device`, in eval mode."""
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype).to(device).eval()
    return tok, model


def chat_prompt(tok, user_text: str, system: str | None = None) -> str:
    """Provided. The exact string an instruct model expects: its chat template, applied.

    The lecture's *A decoder becomes an assistant* slide: an instruct model was tuned on
    conversations written in one exact format, so every message goes through the tokenizer's
    `apply_chat_template` rather than being sent as a bare string.
    """
    messages = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": user_text}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


print("scaffolding ready")


scaffolding ready


### Load the ticket set

Local first, URL fallback, so this works in a clone of the repository and in a Colab tab
with nothing checked out.


In [4]:
# --- load ---------------------------------------------------------------------
DATA_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-05-transformers-llms/data/")
CANDIDATES = [Path("data"), Path("week-05-transformers-llms/data"),
              Path("../data"), Path("../../week-05-transformers-llms/data")]


def data_path(name: str) -> str:
    for base in CANDIDATES:
        if (base / name).exists():
            return str(base / name)
    return DATA_URL + name


tickets = pd.read_csv(data_path("support_tickets.csv"))
GOLD = tickets.category.tolist()
CATEGORIES = ["Shipping", "Billing", "Returns", "Defect", "Account", "Other"]

assert len(tickets) == 60
assert set(tickets.category) == set(CATEGORIES)

print(f"{len(tickets)} tickets")
print(pd.crosstab(tickets.category, tickets.difficulty).loc[CATEGORIES])
print()
print(tickets.head(3).to_string(index=False, max_colwidth=70))


60 tickets
difficulty  ambiguous  clear
category                    
Shipping            3      7
Billing             3      7
Returns             3      7
Defect              3      7
Account             3      7
Other               3      7

  id category difficulty                                                                   text
T001 Shipping      clear Order 88213 was supposed to arrive Tuesday and the tracking hasn't ...
T002 Shipping      clear Tracking says delivered but there is nothing at my door and nothing...
T003 Shipping      clear I paid for two-day shipping on Thursday and the order still hasn't ...


In [5]:
# --- the routing prompt every model sees ---------------------------------------
# Identical for all four systems under comparison. If the prompt differs, the
# comparison measures the prompt, not the model.

SYSTEM_PROMPT = (
    "You are a support-ticket triage assistant for Northwind Outfitters, an online "
    "outdoor-gear retailer.\n"
    "Read the customer message and choose the single best category.\n\n"
    "Categories:\n"
    "- Shipping: where is my order, delivery delays, lost or misdelivered packages, "
    "tracking, delivery address.\n"
    "- Billing: charges, refunds not yet received, invoices, payment methods, promo "
    "codes, tax.\n"
    "- Returns: starting a return or exchange, return labels, return policy, wrong "
    "item or size sent.\n"
    "- Defect: the product arrived damaged or failed in use; quality complaints.\n"
    "- Account: login, password, email address, profile, loyalty points, newsletter "
    "subscription.\n"
    "- Other: anything that does not fit the categories above.\n\n"
    "Answer with exactly one category word and nothing else."
)
print(SYSTEM_PROMPT)


You are a support-ticket triage assistant for Northwind Outfitters, an online outdoor-gear retailer.
Read the customer message and choose the single best category.

Categories:
- Shipping: where is my order, delivery delays, lost or misdelivered packages, tracking, delivery address.
- Billing: charges, refunds not yet received, invoices, payment methods, promo codes, tax.
- Returns: starting a return or exchange, return labels, return policy, wrong item or size sent.
- Defect: the product arrived damaged or failed in use; quality complaints.
- Account: login, password, email address, profile, loyalty points, newsletter subscription.
- Other: anything that does not fit the categories above.

Answer with exactly one category word and nothing else.


### Smoke test — does this machine actually do this?

Provided, and not behind a TODO on purpose. It downloads the smallest of the checkpoints
and generates thirty tokens. If this cell is slow the first time, that is the download; if
it is slow every time, check `device` above.

Read the reply. It is a 362-million-number model, and the answer it gives is your first
evidence about what that buys you.


In [6]:
# --- smoke test ---------------------------------------------------------------
_tok, _model = load_model(MODELS[0])
_prompt = chat_prompt(_tok, "In one sentence: what does an online retailer's support desk do?",
                      system="You are a concise assistant.")
_ids = _tok(_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    _out = _model.generate(**_ids, max_new_tokens=30, pad_token_id=_tok.eos_token_id, **GREEDY)
print(_tok.decode(_out[0][_ids["input_ids"].shape[1]:], skip_special_tokens=True))
free_model(_model)
del _tok, _model, _ids, _out


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

An online retailer's support desk provides assistance to customers with issues related to their online shopping experience.


---

## Task 1 — Two bills, before you download anything

*Lecture slide: **Running a model: two bills, memory and time**. Notebook §4.4.*

**Bill 1, memory to hold the model,** is one multiplication: learned numbers × bytes per
number. **Bill 2, time to write,** is one token at a time, each new token looking back at the
whole text so far — unless the library *remembers* the past instead of re-reading it, which
it may, because a decoder's past never changes.

Three helpers to write, then two provided cells that run them over the three models.


In [7]:
# --- TODO 1a: the memory bill -----------------------------------------------------

def learned_numbers(model) -> int:
    """How many learned numbers the model holds.

    Every parameter tensor `p` in `model.parameters()` holds `p.numel()` numbers; add them up.
    """
    total = 0
    for p in model.parameters():
        total += p.numel()
    # used AI to identify that 'return total' was not needed.
    # TODO
    # todo("learned_numbers")
    


def predicted_gb(n_numbers: int, bytes_per_number: float) -> float:
    """Bill 1, predicted: numbers x bytes per number, in gigabytes (divide by 1e9)."""
    predicted= n_numbers * bytes_per_number
    return predicted
    # TODO
    todo("predicted_gb")


def measured_gb(model) -> float:
    """Bill 1, measured: the bytes the parameters actually occupy right now, in gigabytes.

    Each tensor occupies `p.numel() * p.element_size()` bytes (element_size is 4 at full
    precision, 2 at half). Add them up and divide by 1e9.
    """
    return ( learned_numbers*predicted_gb/1e9)
    # TODO
    todo("measured_gb")


# quick self-check on a tiny stand-in "model": two tensors, 10 and 6 numbers, 4 bytes each
_probe = torch.nn.Sequential(torch.nn.Linear(2, 3), torch.nn.Linear(3, 2))     # 9 + 8 = 17 numbers
if attempt("learned_numbers", learned_numbers, _probe) is not None:
    assert learned_numbers(_probe) == 17, learned_numbers(_probe)
    assert abs(predicted_gb(8_000_000_000, 4) - 32.0) < 1e-9
    assert abs(measured_gb(_probe) - 17 * 4 / 1e9) < 1e-12
    print("Task 1a helpers pass their self-check")


In [8]:
# --- provided: Bill 1 for the three models ---------------------------------------------
# Each model is loaded at full precision (4 bytes), measured, and freed. Then the middle
# model is loaded again at HALF precision (2 bytes), so you can see the second row of the
# lecture's table come true on your own machine.
memory_rows = []
for _name in MODELS:
    _tok, _model = load_model(_name)
    _n = attempt("learned_numbers", learned_numbers, _model)
    if _n is None:
        free_model(_model); del _tok, _model
        break
    memory_rows.append({
        "model": _name.split("/")[-1],
        "learned numbers (M)": round(_n / 1e6, 1),
        "predicted GB @ 4 bytes": round(predicted_gb(_n, 4), 2),
        "predicted GB @ 2 bytes": round(predicted_gb(_n, 2), 2),
        "predicted GB @ 1/2 byte (4-bit)": round(predicted_gb(_n, 0.5), 2),
        "measured GB (as loaded)": round(measured_gb(_model), 2),
    })
    free_model(_model); del _tok, _model

memory_table = pd.DataFrame(memory_rows) if len(memory_rows) == len(MODELS) else None
if memory_table is not None:
    print(memory_table.to_string(index=False))
    _tok, _half = load_model(MODELS[1], dtype=torch.bfloat16)
    print(f"\n{MODELS[1].split('/')[-1]} loaded at HALF precision: measured {measured_gb(_half):.2f} GB"
          f"  (predicted @ 2 bytes: {memory_table.loc[1, 'predicted GB @ 2 bytes']:.2f} GB)")
    free_model(_half); del _tok, _half


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [9]:
# --- TODO 1b: the time bill --------------------------------------------------------

def time_to_write(model, tok, prompt: str, n_new: int = 32, use_cache: bool = True) -> dict:
    """Write exactly `n_new` tokens after `prompt`, greedily, and time it.

    Returns {"prompt_tokens": int, "seconds": float, "tokens_per_second": float}.

    Steps
      1. ids = tok(prompt, return_tensors="pt").to(device)
      2. warm up: one short generate (4 tokens) so the first call's start-up cost is not timed
      3. sync(); t0 = time.perf_counter()
      4. model.generate(**ids, max_new_tokens=n_new, min_new_tokens=n_new, use_cache=use_cache,
                        pad_token_id=tok.eos_token_id, **GREEDY)   -- inside torch.no_grad()
      5. sync(); seconds = time.perf_counter() - t0
      6. return the dict; prompt_tokens is ids["input_ids"].shape[1]

    `use_cache=True` is the library remembering the past; `use_cache=False` makes it
    re-read the whole text for every new token.
    """
#1
    ids = tok(prompt, return_tensors='pt').to(device)
#2
    with torch.no_grad():
      model.generate(**ids, max_new_tokens=4, use_cache=use_cache,
                   pad_token_id=tok.eos_token_id, **GREEDY)
#3
    sync() ;t0= time.perf_counter()
#4
    model.generate(**ids, max_new_tokens=n_new, min_new_tokens=n_new, use_cache=use_cache,pad_token_id=tok.eos_token_id, **GREEDY)
#5
    sync(); seconds= time.perf_counter() - t0
#6
    return{ "prompt_tokens": ids["input_ids"].shape[1],
           "seconds":seconds,
           "tokens_per_second":n_new / seconds}

    
    # TODO
    todo("time_to_write")


In [10]:
# --- provided: Bill 2 -- one token at a time, remembering the past or not ---------------
# A short prompt (one ticket) and a long one (the same ticket after a long transcript),
# each written to 32 new tokens, with the past remembered and with it re-read every step.
FILLER = ("Earlier in this conversation the customer described a delayed order, the agent "
          "quoted the carrier's tracking status and the exception code, and the customer "
          "asked twice when the parcel would be rescanned at the regional hub. ")
TICKET = tickets.loc[tickets.id == "T002", "text"].item()

if not need(attempt("time_to_write", lambda: time_to_write)):
    _tok, _model = load_model(MODELS[1])
    SHORT = chat_prompt(_tok, TICKET + "\n\nReply in two sentences.", system="You are a support agent.")
    LONG = chat_prompt(_tok, FILLER * 22 + "\n\n" + TICKET + "\n\nReply in two sentences.",
                       system="You are a support agent.")
    cache_rows = []
    for _label, _prompt in (("short", SHORT), ("long", LONG)):
        on = attempt("time_to_write", time_to_write, _model, _tok, _prompt, 32, True)
        off = attempt("time_to_write", time_to_write, _model, _tok, _prompt, 32, False)
        if on is None or off is None:
            cache_rows = None
            break
        cache_rows.append({"text so far": f"{_label} ({on['prompt_tokens']:,} tokens)",
                           "re-reads everything (s)": round(off["seconds"], 2),
                           "remembers the past (s)": round(on["seconds"], 2),
                           "speed-up": f"{off['seconds'] / on['seconds']:.1f}x",
                           "tokens/s with the past remembered": round(on["tokens_per_second"], 1)})
    free_model(_model); del _tok, _model
    if cache_rows:
        cache_table = pd.DataFrame(cache_rows)
        print(f"{MODELS[1].split('/')[-1]}, 32 new tokens\n")
        print(cache_table.to_string(index=False))

        # and the writing speed of all three sizes, on the short prompt, past remembered
        speed_rows = []
        for _name in MODELS:
            _tok, _model = load_model(_name)
            r = time_to_write(_model, _tok, chat_prompt(_tok, TICKET + "\n\nReply in two sentences.",
                                                        system="You are a support agent."), 32, True)
            speed_rows.append({"model": _name.split("/")[-1], "tokens/s": round(r["tokens_per_second"], 1)})
            free_model(_model); del _tok, _model
        speed_table = pd.DataFrame(speed_rows)
        print()
        print(speed_table.to_string(index=False))


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


                model  tokens/s
SmolLM2-360M-Instruct       4.8
Qwen2.5-0.5B-Instruct       4.1
Qwen2.5-1.5B-Instruct       1.4


### Your answer — Task 1

1. A "0.5B" model does describes a model at 0.5 Billion Parameters. At full precision(four bytes per number), this model would take about 2GBs on memory. For a 8GB latop, this could run smoothly. However, a 1.5B model would take around 6GBs which would slow the laptop. This is because the remaining 2GBs still has to run other processes( Google Chrome, SYstems,...) 
+ Models can be stored on memory at smaller bytes per number( half-precision, 8-bit,4-bit).At 4-bit quantisation, high-grade models can still be ran, but it will focus more on reasoning than formatting.

2.In the two models in the cache, speedup can be bigger on the longer sentence due to more unimportant words, as the attention mechanism grades based on relevance to task(and map them through softmax) instead of count. As a decoder, "Re reads everything" is a mechanism to generate new tokens base on context. At the same time,"Renembers the past" would be the constant weight that is transferred through each layer of the multi-head attention, nudging the model to a specific context(e.g shipping). Even as a decoder, the context keeps the model on task. This could be the reason why most commercial models are decoders. Their parameters are large enough that embeddings cover most simple queries. 

3.'Memory' grows with the length of the conversation when 'time-to-write' stays fixed. Firstly, the model takes a certain space on hardware when it loads the pre-trained weights(2GB for 0.5 full precision, less for 4-bit). Secondly, as the session progresses, memory grows with every token of a conversation, around 12KB per token for the 0.5B model. (Metrics taken from Week05 Lecture Notes)



---

## Task 2 — What the model does with a message

*Lecture slides: **A decoder becomes an assistant in two stages** and **Choosing the next
token**. Notebook §4.2 and §5.7.*

One helper to write — the four-line loop from the lecture, wrapped by `generate` — then two
provided cells. The first hands the same ticket to the **base** model and to the **instruct**
model built from it. The second hands one prompt to the instruct model under five decoding
settings.


In [11]:
# --- TODO 2a: get a reply -----------------------------------------------------------

def reply(model, tok, prompt: str, max_new_tokens: int = 60, **decoding) -> str:
    """Tokenize `prompt`, generate up to `max_new_tokens`, return ONLY the new text.

    `decoding` is either GREEDY or one of the SAMPLE(...) dicts.

    Steps
      1. ids = tok(prompt, return_tensors="pt").to(device)
      2. out = model.generate(**ids, max_new_tokens=max_new_tokens,
                              pad_token_id=tok.eos_token_id, **decoding)   -- inside torch.no_grad()
      3. new = out[0][ids["input_ids"].shape[1]:]     -- the tokens AFTER the prompt
      4. return tok.decode(new, skip_special_tokens=True).strip()
    """
    ids= tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
      out= model.generate(**ids,max_new_tokens=max_new_tokens,pad_token_id=tok.eos_token_id,**decoding)
      new= out[0][ids["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip()


In [12]:
# --- provided: base vs instruct, same weights before and after the second stage ---------
# The base model is a text continuer: it gets the ticket as bare text. The instruct model
# gets the same ticket through its chat template.
if not need(attempt("reply", lambda: reply)):
    _btok, _base = load_model(BASE)
    print(f"TICKET: {TICKET}\n")
    print(f"--- {BASE}  (bare text in, greedy) ---")
    _text = attempt("reply", reply, _base, _btok, TICKET + "\n\n", 60, **GREEDY)
    if _text is not None:
        print(_text)
    free_model(_base); del _btok, _base

    _tok, _inst = load_model(MODELS[1])
    print(f"\n--- {MODELS[1]}  (chat template, greedy) ---")
    _text = attempt("reply", reply, _inst, _tok, chat_prompt(_tok, TICKET, system="You are a support agent."),
                    60, **GREEDY)
    if _text is not None:
        print(_text)


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

TICKET: Tracking says delivered but there is nothing at my door and nothing with my neighbours.

--- Qwen/Qwen2.5-0.5B  (bare text in, greedy) ---
Does it follow that "I have a lot of mail to deliver."?
Options are:
 (a). yes
 (b). it is not possible to tell
 (c). no
(b).


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


--- Qwen/Qwen2.5-0.5B-Instruct  (chat template, greedy) ---
I'm sorry to hear that you're experiencing this issue. It sounds like there may be a problem with the delivery service or the delivery person's location. Here are a few steps you can take to try and resolve the issue:

1. Check the delivery service's website or app for any updates or


In [13]:
# --- provided: five ways of choosing the next token --------------------------------------
# Same model, same prompt. Greedy twice; temperature 0.7 with top-p 0.9 under two seeds;
# temperature 1.5 with no truncation.
if not need(attempt("reply", lambda: reply)):
    DRAFT = chat_prompt(_tok, "Write a two-sentence reply to this customer:\n\n" + TICKET,
                        system="You are a Northwind Outfitters support agent.")
    runs = [("greedy, run 1", GREEDY, None), ("greedy, run 2", GREEDY, None),
            ("T=0.7, top-p 0.9, seed 1", SAMPLE(0.7, 0.9), 1),
            ("T=0.7, top-p 0.9, seed 2", SAMPLE(0.7, 0.9), 2),
            ("T=1.5, no truncation, seed 1", SAMPLE(1.5, 1.0), 1)]
    for label, settings, seed in runs:
        if seed is not None:
            torch.manual_seed(seed)
        text = attempt("reply", reply, _inst, _tok, DRAFT, 60, **settings)
        if text is None:
            break
        print(f"[{label}]\n{text}\n")
    free_model(_inst); del _tok, _inst


[greedy, run 1]
I'm sorry to hear that you're experiencing issues with your order. Please let me know if you need any assistance with tracking or if there's anything else I can help you with.

[greedy, run 2]
I'm sorry to hear that you're experiencing issues with your order. Please let me know if you need any assistance with tracking or if there's anything else I can help you with.

[T=0.7, top-p 0.9, seed 1]
I'm sorry to hear that. Can you provide me with your contact information so I can assist you further?

[T=0.7, top-p 0.9, seed 2]
I apologize, but I don't have access to your specific delivery information. Can you please provide me with the tracking number so I can check for delivery status?

[T=1.5, no truncation, seed 1]
I'm sorry to hear that the product is failing to empty without their grasp itDMETHOD corporation design product efficiently prioritize detail一点点 caputo customer detail limeos bc related"wcu proof with discretion point合法checkBoxTexasample opti Production Burger社团

### Your answer — Task 2

1. Both models have the same learned numbers. However, with different Temperatures, the results are varied. Temperature can change the logit distances for probabilities of the next token. The second stage of training addressed repetition, and helps the engineer select how much variable(personality) a model can have. 

2. Temperature determine the model's variability. At a low temeperature or greedy(T close to 0), the next token is always the top word. This explains why the two answers are exactly identical as there are little variability. At the temperature of 1.5, the answer is more varied as logit distances shrink and secondary toekns get more chance of being selected. 

3. A greedy setting is better to route tickers. This is because for classification(normally encoders), a high F-1 score is favorable and repetition is acceptable. For replies however, a temperature score of 1 is good. This promotes a wider range of responses which could satisfy customers( some would not want robotic responses). A score of 1 is picked over 1.5 because while responses at 1, the responses are varied but not too unpredictable(risk of being unprofessional or off-topic).

---

## Task 3 — Does it actually do the job?

*Lecture slide: **Small vs large is an engineering decision**. Notebook §6.2–6.3.*

The lecture measured three model sizes on a 24-task set. Here you do the same on the desk's
own tickets: 60 of them, hand-labelled into six queues, ten per queue, with a `difficulty`
column that separates the ones a careful reader answers on sight (`clear`) from the ones that
need the desk's tie-break rules (`ambiguous`). The rules are in
[`../data/README.md`](../data/README.md); **read them before you argue with a label.**

Two helpers to write, then the loop from the lecture notebook's `run_task_set`, then
provided tables.


In [14]:
# --- TODO 3a: the message, and the parser -----------------------------------------------

def build_messages(text: str) -> list[dict]:
    """The two-message chat sent to every model: SYSTEM_PROMPT, then the ticket text."""
    return [{"role":"system","content":SYSTEM_PROMPT},{"role":'user', "content":text}]
    
    # TODO
    todo("build_messages")


def parse_label(raw: str):
    """Turn a reply into one of CATEGORIES, or None if no category word appears.

    Case-insensitive; the FIRST category word that appears anywhere in the reply wins
    ("Category: Billing." -> "Billing"). A reply with no category word is invalid: return
    None. In production an invalid reply cannot be routed, so it counts as wrong.
    """
    lower=raw.lower()
    best_pos= None
    best_name= None

    for name in CATEGORIES:#Used AI to restructure and provide sample code
        pos=lower.find(name.lower())
        if pos !=-1 and (best_pos is None or pos < best_pos):
            best_pos= pos
            best_name=name
    return best_name
    
    # TODO
    todo("parse_label")


for _s, _want in [("Shipping", "Shipping"), (" billing ", "Billing"), ("**Account**", "Account"),
                  ("Category: Returns.", "Returns"), ("This is a Defect, not Billing", "Defect"),
                  ("I'm sorry to hear that", None), ("", None)]:
    _got = attempt("parse_label", parse_label, _s)
    if _got is None and _want is not None:
        break
    assert _got == _want, (_s, _got, _want)
else:
    print("parse_label passes its self-check")


parse_label passes its self-check


In [15]:
# --- TODO 3b: run one model over all sixty tickets ------------------------------------------

def classify_all(model, tok, max_new_tokens: int = 8) -> dict:
   """Greedy-classify every ticket. Returns {"raw": [str], "new_tokens": [int], "seconds": [float]}.

    For each ticket text in `tickets.text`:
      1. prompt = tok.apply_chat_template(build_messages(text), tokenize=False,
                                          add_generation_prompt=True)
      2. ids = tok(prompt, return_tensors="pt").to(device)
      3. sync(); t0 = time.perf_counter()
         out = model.generate(**ids, max_new_tokens=max_new_tokens,
                              pad_token_id=tok.eos_token_id, **GREEDY)   -- inside torch.no_grad()
         sync(); seconds = time.perf_counter() - t0
      4. new = out[0][ids["input_ids"].shape[1]:]
         raw = tok.decode(new, skip_special_tokens=True).strip()
      5. append raw, len(new) and seconds to the three lists
   """
   raws, new_tokens, secs = [], [], []
   for text in tickets.text:
      #1
      prompt= tok.apply_chat_template(build_messages(text), tokenize= False, add_generation_prompt=True)
      #2
      ids= tok(prompt, return_tensors="pt").to(device)
      #3
      sync(); t0 = time.perf_counter()
#4    
      with torch.no_grad():
         out = model.generate(**ids, max_new_tokens=max_new_tokens,
                                    pad_token_id=tok.eos_token_id, **GREEDY) 
#5
         sync(); seconds = time.perf_counter() - t0
#6
         new = out[0][ids["input_ids"].shape[1]:]
         raw = tok.decode(new, skip_special_tokens=True).strip()
#7
         raws.append(raw)
         secs.append(seconds)
         new_tokens.append(len(new)) 
   return {"raw": raws, "new_tokens": new_tokens, "seconds": secs}


In [16]:
# --- provided: all three models over all sixty tickets ----------------------------------------
gen_results = {}
for _name in MODELS:
    _tok, _model = load_model(_name)
    _out = attempt("classify_all", classify_all, _model, _tok)
    free_model(_model); del _tok, _model
    if _out is None:
        gen_results = None
        break
    gen_results[_name] = _out
    print(f"{_name:42s} {sum(_out['seconds']):6.1f} s   first replies: {_out['raw'][:4]}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

HuggingFaceTB/SmolLM2-360M-Instruct         248.8 s   first replies: ['Shipping', 'Shipping', 'Shipping', 'Yes, you can change the delivery address']


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen/Qwen2.5-0.5B-Instruct                  224.6 s   first replies: ['Tracking', 'Delivery address', 'Shipping', 'Billing']


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen/Qwen2.5-1.5B-Instruct                  764.9 s   first replies: ['Shipping', 'Shipping', 'Shipping', 'Shipping']


In [17]:
# --- provided: the metrics table and the per-queue table ---------------------------------------

def summarise(out: dict) -> dict:
    preds = [parse_label(r) for r in out["raw"]]
    ok = [p == g for p, g in zip(preds, GOLD)]
    diff = tickets.difficulty.tolist()
    return {
        "accuracy": round(float(np.mean(ok)), 3),
        "invalid replies": round(float(np.mean([p is None for p in preds])), 3),
        "acc clear": round(float(np.mean([o for o, d in zip(ok, diff) if d == "clear"])), 3),
        "acc ambiguous": round(float(np.mean([o for o, d in zip(ok, diff) if d == "ambiguous"])), 3),
        "mean new tokens": round(float(np.mean(out["new_tokens"])), 2),
        "s per ticket": round(float(np.mean(out["seconds"])), 3),
    }


if not need(gen_results):
    metrics = pd.DataFrame([{"model": n.split("/")[-1], **summarise(o)} for n, o in gen_results.items()])
    print(metrics.to_string(index=False))

    # accuracy per queue: the table the lecture said not to skip -- one overall number
    # hides a rare-queue failure
    per_queue = {}
    for n, o in gen_results.items():
        preds = [parse_label(r) for r in o["raw"]]
        per_queue[n.split("/")[-1]] = {
            c: round(float(np.mean([p == g for p, g in zip(preds, GOLD) if g == c])), 2) for c in CATEGORIES}
    print("\naccuracy per queue (10 tickets each)\n")
    print(pd.DataFrame(per_queue).T.to_string())


                model  accuracy  invalid replies  acc clear  acc ambiguous  mean new tokens  s per ticket
SmolLM2-360M-Instruct     0.133            0.250      0.119          0.167             4.72         4.147
Qwen2.5-0.5B-Instruct     0.450            0.083      0.476          0.389             2.38         3.743
Qwen2.5-1.5B-Instruct     0.633            0.000      0.738          0.389             2.15        12.749

accuracy per queue (10 tickets each)

                       Shipping  Billing  Returns  Defect  Account  Other
SmolLM2-360M-Instruct       0.8      0.0      0.0     0.0      0.0    0.0
Qwen2.5-0.5B-Instruct       0.5      0.3      0.8     0.9      0.2    0.0
Qwen2.5-1.5B-Instruct       0.7      0.8      1.0     0.8      0.4    0.1


In [18]:
# --- provided: the confusion matrix and every wrong reply, for your best local model ---------
if not need(gen_results):
    BEST_LOCAL = MODELS[-1]          # change this if a different model scored highest
    best = [parse_label(r) for r in gen_results[BEST_LOCAL]["raw"]]
    filled = [p if p is not None else "<invalid>" for p in best]
    labels = CATEGORIES + ["<invalid>"]
    cm = confusion_matrix(GOLD, filled, labels=labels)
    print(f"Confusion matrix -- {BEST_LOCAL}  (rows = gold queue, columns = predicted)\n")
    print(pd.DataFrame(cm, index=labels, columns=labels).loc[CATEGORIES].to_string())

    print("\n\nEvery ticket it got wrong:\n")
    for i, (p, g) in enumerate(zip(filled, GOLD)):
        if p != g:
            t = tickets.iloc[i]
            print(f"{t.id}  {t.difficulty:<9s} gold={g:<9s} pred={p:<10s} raw={gen_results[BEST_LOCAL]['raw'][i]!r}")
            print(f"        {t.text[:100]}")


Confusion matrix -- Qwen/Qwen2.5-1.5B-Instruct  (rows = gold queue, columns = predicted)

          Shipping  Billing  Returns  Defect  Account  Other  <invalid>
Shipping         7        1        1       1        0      0          0
Billing          0        8        0       0        2      0          0
Returns          0        0       10       0        0      0          0
Defect           0        0        2       8        0      0          0
Account          0        3        2       0        4      1          0
Other            2        0        7       0        0      1          0


Every ticket it got wrong:

T007  clear     gold=Shipping  pred=Returns    raw='Returns'
        My order shipped in two boxes and only one arrived. The sleeping bag is missing.
T008  ambiguous gold=Shipping  pred=Billing    raw='Billing'
        I was charged for expedited shipping but the package took nine days. I'd like the shipping fee back.
T010  ambiguous gold=Shipping  pred=Defect     raw='Defe

### Your answer — Task 3


1. Accuracy rises with size. We saw the most gain switching from a mini model to 0.32 for a medium model and moderate gain of 0.18 to the large model. This corresponds to the law of Diminishing returns, and tradeoffs needs to be weighed for the selection of the last two model.  

2. The best model is Qwen 1.5B. It's hardest queue is 'Others' at 0.1 accuracy. On both incorrect replies, the tickets that were falsely classified as 'Other' had ambiguous signals. The reason 'Others' is wrong could be due to two-intent sentences.

3. The medium model had low 'invalid replies', 'mean new tokens' and the lowest 's per ticket'. The smalles model is not the fastert per ticket. This could be because the lowest model as a decoder did not have enough trained parameters and hence took longer(4s) to process the task. Similarly, the 1.5B model might have too many parameters which made it difficult to narrow down(13s). 

4. One number that hides expensive mistakes could be 'invalid'. This mistake can lead to incorrect queries, and would require a human-in-the-loop to comeback and validate these tickets. 

---

## Task 4 — Small vs large, and your recommendation

*Lecture slides: **Small vs large is an engineering decision** and the wrap-up case,
**Design Northwind's support workflow**. Notebook §6.4–6.5.*

You do not have an API key yet — those arrive in Week 6 — so the frontier side is supplied
as a **fixture**: a reply string for each of the 60 tickets, scored exactly the way the local
models are scored. The reply strings were **authored, not sampled from a live model**; the
token counts are real, so the cost arithmetic is real, but the accuracy it produces is an
illustration of the procedure, not a measurement of any product. Your memo must say so.


In [19]:
# --- provided: load the frontier fixture --------------------------------------
import urllib.request

_fp = data_path("frontier_cached.json")
if Path(_fp).exists():
    frontier = json.loads(Path(_fp).read_text())
else:
    with urllib.request.urlopen(_fp) as fh:
        frontier = json.loads(fh.read())

print(json.dumps(frontier["meta"], indent=2))
frontier_raw = [frontier["responses"][i]["reply"] for i in tickets.id]
print("\nfirst eight replies:", frontier_raw[:8])


{
  "kind": "fixture",
  "model_label": "frontier-mini (stand-in)",
  "note": "Authored reply strings, NOT sampled from a live model. Token counts are real tiktoken o200k_base counts over the exact strings above. No latency is recorded because none was measured. Set OPENAI_API_KEY and re-run the notebook's live cell to replace this with a real transcript.",
  "encoding": "o200k_base",
  "system_prompt_sha256": "9a64918d1a1a39199253a5ec1e438b16da40183e20e5d326ba26ed22af4b9839",
  "system_prompt_tokens": 154,
  "n_tickets": 60,
  "n_agreeing_with_gold": 52
}

first eight replies: ['Shipping', 'Shipping', 'Shipping.', 'Shipping', 'Shipping', 'Shipping', 'Shipping', 'Billing']


In [21]:
# --- TODO 4a: the API bill ------------------------------------------------------------
# List prices for a frontier "mini" tier, in US dollars per MILLION tokens. Inputs to your
# analysis, not facts about the universe: check the provider's pricing page and record the
# date you checked in your memo.
PRICE_IN_PER_M = 10 #Claude Fable5.1 : $10 per Mtoken . Date: 09/29/26
PRICE_OUT_PER_M = 50 #Claude Fable 5.1:  $50 per token
TICKETS_PER_WEEK = 2400


def usd_cost(input_tokens: int, output_tokens: int,
             price_in_per_m: float = PRICE_IN_PER_M, price_out_per_m: float = PRICE_OUT_PER_M) -> float:
    """Dollar cost of a request or a batch: tokens / 1,000,000 x price, for input and output."""
    # TODO
    in_price= input_tokens / 1_000_000 * PRICE_IN_PER_M
    out_price= output_tokens /1_000_000 * PRICE_OUT_PER_M 
    return in_price+out_price
    todo("usd_cost")


_in = sum(v["prompt_tokens"] for v in frontier["responses"].values())
_out = sum(v["completion_tokens"] for v in frontier["responses"].values())
_c60 = attempt("usd_cost", usd_cost, _in, _out)
if _c60 is not None:
    per_ticket_usd = _c60 / len(tickets)
    print(f"fixture totals: {_in:,} input tokens, {_out:,} output tokens for {len(tickets)} tickets")
    print(f"cost per ticket : ${per_ticket_usd:.6f}")
    print(f"cost per week   : ${per_ticket_usd * TICKETS_PER_WEEK:.2f}   ({TICKETS_PER_WEEK:,} tickets)")
    print(f"cost per year   : ${per_ticket_usd * TICKETS_PER_WEEK * 52:.2f}")


fixture totals: 10,351 input tokens, 76 output tokens for 60 tickets
cost per ticket : $0.001788
cost per week   : $4.29   (2,400 tickets)
cost per year   : $223.20


In [22]:
# --- provided: the comparison table -------------------------------------------------------
# The local machine is already in the ops room, so its line is machine time, not dollars:
# seconds per ticket x 2,400 tickets, in minutes per week.
if not need(gen_results, _c60):
    rows = []
    for n, o in gen_results.items():
        m = summarise(o)
        rows.append({"system": n.split("/")[-1], "where the text goes": "the ops-room machine",
                     "accuracy": m["accuracy"], "acc clear": m["acc clear"], "acc ambiguous": m["acc ambiguous"],
                     "invalid replies": m["invalid replies"], "s per ticket": m["s per ticket"],
                     "cost per week": f"{m['s per ticket'] * TICKETS_PER_WEEK / 60:.1f} machine-minutes"})
    f_out = {"raw": frontier_raw, "new_tokens": [frontier["responses"][i]["completion_tokens"] for i in tickets.id],
             "seconds": [float("nan")] * len(tickets)}
    fm = summarise(f_out)
    rows.append({"system": "frontier (FIXTURE)", "where the text goes": "a third party's servers",
                 "accuracy": fm["accuracy"], "acc clear": fm["acc clear"], "acc ambiguous": fm["acc ambiguous"],
                 "invalid replies": fm["invalid replies"], "s per ticket": "not measured",
                 "cost per week": f"${per_ticket_usd * TICKETS_PER_WEEK:.2f}"})
    comparison = pd.DataFrame(rows)
    print(comparison.to_string(index=False))
    print("\nThe frontier row's accuracy is a fixture and its latency was never measured. Say both in the memo.")


               system     where the text goes  accuracy  acc clear  acc ambiguous  invalid replies s per ticket         cost per week
SmolLM2-360M-Instruct    the ops-room machine     0.133      0.119          0.167            0.250        4.147 165.9 machine-minutes
Qwen2.5-0.5B-Instruct    the ops-room machine     0.450      0.476          0.389            0.083        3.743 149.7 machine-minutes
Qwen2.5-1.5B-Instruct    the ops-room machine     0.633      0.738          0.389            0.000       12.749 510.0 machine-minutes
   frontier (FIXTURE) a third party's servers     0.867      0.976          0.611            0.000 not measured                 $4.29

The frontier row's accuracy is a fixture and its latency was never measured. Say both in the memo.


### Your memo — Task 4

*Replace this text.* **300–450 words**, addressed to the support operations director. A
decision memo, not a lab report. Use the three headings from the lecture's workflow case:

**Choose.** Recommend one of: run a local model, buy the API, run a hybrid, or ship nothing
yet. Commit to one. Name the model and the decoding setting, and cite at least **four
numbers you measured**, each with the model or condition it came from.

**Evaluate.** Say which errors matter and what your 60-ticket set can and cannot tell the
director. Name the one measurement you would take next. State plainly which of your numbers
are measurements and which came from the fixture.

**Control.** Say when a person must approve: which queues, which ambiguous tickets, which
invalid replies. Name the conditions under which your recommendation flips — at least ticket
volume, the mix of clear versus ambiguous traffic, and the legal position on sending customer
text to a third party.


To: Support Operations Director, Northwind Outfitters
From: Long Trinh
Date: 09/29/26
Subject: Ticket Selection Model

Based on the company's need, using a frontier model's API(Claude Fable 5.1) is the most cost efficient model. At 09/29/26, for over 2400 tickets, this ticket costs $10 per M input tokens and $50 per M output tokens. For deployment, this model costs $4.29 a week or $223.20 a month. This model performs well based on fixture numbers: 0.867 accuracy, 0.976 clear, 0.611 acc ambiguous and time per ticket is never measured. However, it's accuracy metric is 0.20 better than the local model. With 2400 tickets, that is 480 more accurate tickets. The best running local model, Qwen 1.5B, costs 510 machine-minutes per week. For the organization, the cost of the API key is reasonable for a higher accuracy and runtime.  While the local model does not cost money, the measurements does not justify running the model. With its accuracy at 0.63, acc clear at 0.738, acc ambiguous at 0.389 and s per ticket at 12.749 seconds makes it inefficient for ranking. The model would take up to 510 machine minutes per week. Hence, a frontier API key is optimal choice. 

One of the errors to consider is for multi-intent sentences. This 60-ticket set demonstrates how many intents can decrease accuracy. With more intents in a ticket, the model can perform less accurately.One measurement that I could take next is ambiguous%. This measurement can help rank the model's performance better and could potentially determine that some models are better at handling two-intent tickets. This could then help assist model selection. This measurement helps explain the gap in detail that the 60-ticket set cannot tell us. 

For this model to work well in a business, it would also need a human in the loop. This model performs well due to it's ability to classify tickets for also complex cases, and it's ability to automate 2400 tickets in a week. However, there is a 0.15 gap of accuracy. This means that even for automation, there are certain cases when a human actor is involved in resolving ambiguous tickets. In the long term, this data and trace of correcting tickets can be used to build a new classification head for the project. Automation is prefered over human validation in the case of single-intent tickets. Most models performed well for classifying clear ticket intents. This could massively cut down human-labor and processing time. 